last modified date : 2026.07   
제작 : 모두의연구소 퍼실팀

# Day 2 — Advanced·Modular RAG + RAGAS 평가

In [1]:
# Colab pre-installed langchain 0.3 / ragas 0.1~0.4 를 ragas 0.2.10 호환 조합으로 정리합니다.
# 처음 실행 시 약 3~5분 걸립니다. 진행률 출력을 보면서 기다리세요 (멈춘 게 아닙니다).

# 1) 기존 langchain / ragas 패키지 제거 — 버전 충돌로 인한 pip resolver 백트래킹 방지
!pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma

# 2) 0.2 시리즈 패치 버전까지 핀 설치 — resolver 부담 최소화 (-q 제거해서 진행률 보이게)
!pip install --no-cache-dir \
    "ragas==0.2.10" \
    "langchain==0.2.17" \
    "langchain-core==0.2.43" \
    "langchain-community==0.2.19" \
    "langchain-openai==0.1.25" \
    "langchain-text-splitters==0.2.4" \
    "langchain-chroma==0.1.4" \
    pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas

Found existing installation: ragas 0.2.10
Uninstalling ragas-0.2.10:
  Successfully uninstalled ragas-0.2.10
Found existing installation: langchain 0.2.17
Uninstalling langchain-0.2.17:
  Successfully uninstalled langchain-0.2.17
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.19
Uninstalling langchain-community-0.2.19:
  Successfully uninstalled langchain-community-0.2.19
Found existing installation: langchain-openai 0.1.25
Uninstalling langchain-openai-0.1.25:
  Successfully uninstalled langchain-openai-0.1.25
Found existing installation: langchain-text-splitters 0.2.4
Uninstalling langchain-text-splitters-0.2.4:
  Successfully uninstalled langchain-text-splitters-0.2.4
Found existing installation: langchain-chroma 0.1.4
Uninstalling langchain-chroma-0.1.4:
  Successfully uninstalled langchain-chroma-0.1.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
# chromadb 익명 통계 전송 끄기 — posthog SDK 인자 충돌로 ERROR 로그가 뜨는 것 방지
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import nest_asyncio
nest_asyncio.apply()  # RAGAS가 Colab의 비동기 이벤트 루프와 충돌하지 않도록

In [3]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [4]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken, random

tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# 1) 데이터셋 로드 + 2000개 샘플링 + context 중복 제거 → unique 약 800개
#    (Vector DB 가 크면 Reranker 의 정밀도 개선 효과가 더 또렷하게 보입니다.
#     인덱싱 토큰 비용 약 0.01 USD 추가)
raw_ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=42).select(range(2000))

unique = {}
for ex in raw_ds:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique.items()]

# 2) chunk 단위로 분할 (KorQuAD context는 짧지만 길이 균질화를 위해 splitter 사용)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = splitter.split_documents(context_docs)

# 3) Embedding & Chroma 적재 — chunk 약 800개를 한 번에 넣으면 chromadb 의 batch limit
#    (Colab 환경에서 보통 5461) 또는 OpenAI rate limit 에 걸릴 수 있어
#    100개씩 배치로 add_documents 합니다.
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i+BATCH])

# 4) Retriever (Naive: similarity)
naive_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 5) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"베이스라인 준비 완료 — unique context: {len(context_docs)}, chunks: {len(docs)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — unique context: 847, chunks: 1264


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))

Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


A: 대중교통체계입니다.


In [6]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년에 서울시장으로서 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 재임 중에 이루어진 전반적인 개선 내용은 어떤 것들이 있습니까?']


검색된 문서 수: 5
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


In [7]:
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    scores = defaultdict(float)
    docs_by_key = {}

    for docs in results_per_query:
        for rank, doc in enumerate(docs):
            key = doc.page_content
            scores[key] += 1.0 / (k + rank + 1)
            docs_by_key[key] = doc

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs_by_key[k] for k, _ in ranked[:top_k]]

In [8]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])

가상 답변(HyDE):
 2004년 이명박이 서울시장으로 재직하던 시절, 그는 서울시의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 특히, 그는 '서울시 교통체계 개선 종합계획'을 수립하여 대중교통의 효율성을 높이고, 도로 혼잡을 줄이기 위한 다양한 정책을 시행했습니다. 이 과정에서 지하철 노선 확장과 버스 전용차선 도입, 그리고 자전거 도로의 확충 등이 이루어졌습니다. 이러한 노력은 서울시민의 교통 편의를 증대시키고, 대기오염 문제를 완화하는 데 기여했습니다. 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


post-retrieval 강화 — Cross-Encoder Reranking (multilingual)


In [9]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


In [10]:
def advanced_rag(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)
    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    top = rerank(question, candidates, top_k=3)
    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)

Advanced RAG 답변:
 대중교통체계입니다.


In [14]:
# Self-RAG : retrieve 판단 + 자가 비평 + HyDE 재시도
# (1) 검색 필요성 판단 프롬프트 — TODO 1
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    "사용자의 질문이 외부 문서 검색이 필요한지 판단하세요. 일반 상식, 계산, 정의 등 자체 지식으로 답변할 수 있다면 'NO', 외부 정보가 필요하다면 'YES'라고 단 한 단어로만 대답하세요.\n\n질문: {question}"
)

# (2) 답변 자가 비평 프롬프트 — TODO 2
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    "다음 [문서]를 바탕으로 [답변]이 작성되었는지 평가하세요. 답변이 문서의 내용으로 충분히 뒷받침된다면 'SUPPORTED', 뒷받침되지 않거나 무관하다면 'NOT_SUPPORTED'라고 단 한 단어로만 대답하세요.\n\n[문서]\n{context}\n\n[답변]\n{answer}"
)


def self_rag(question, max_retries=1, verbose=True):
    decision = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}).strip().upper()
    if verbose:
        print(f"[1] Retrieve 필요? -> {decision}")

    if decision.startswith("NO"):
        ans = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return ans, []

    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)

    for attempt in range(max_retries + 1):
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        critique = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "answer": answer}).strip().upper()
        if verbose:
            print(f"[3] 시도 {attempt+1} — 자가 비평: {critique}")

        if "NOT" not in critique:
            return answer, docs

        if attempt < max_retries:
            hyp = hyde_generator.invoke({"question": question})
            docs = db.similarity_search(hyp, k=3)
            if verbose:
                print("[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색")

    return answer, docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)

[1] Retrieve 필요? -> YES
[3] 시도 1 — 자가 비평: NOT_SUPPORTED
[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색
[3] 시도 2 — 자가 비평: NOT_SUPPORTED

=== Self-RAG 최종 답변 ===
대중교통체계입니다.


RAGAS 평가용 데이터셋 만들기

In [16]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20  # 평가 질문 수. 표본 분산을 줄이려 20개로 설정. 줄이려면 5~10.
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# Naive RAG 로 답변 + 컨텍스트 수집
naive_answers, naive_contexts = [], []
for q in questions:
    ctx = naive_retriever.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers.append(a)
    naive_contexts.append([d.page_content for d in ctx])

# Advanced RAG 로 답변 + 컨텍스트 수집
adv_answers, adv_contexts = [], []
for q in questions:
    a, ctx = advanced_rag(q)
    adv_answers.append(a)
    adv_contexts.append([d.page_content for d in ctx])

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")

데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [17]:
from datasets import Dataset

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

In [18]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb  = OpenAIEmbeddings(model="text-embedding-3-small")
metrics    = [faithfulness, answer_relevancy,
              context_precision, context_recall]

print("=== Naive RAG 채점 ===")
naive_result = evaluate(naive_ds, metrics=metrics,
                        llm=judge_llm, embeddings=judge_emb,
                        raise_exceptions=False)

print("=== Advanced RAG 채점 ===")
adv_result = evaluate(adv_ds, metrics=metrics,
                      llm=judge_llm, embeddings=judge_emb,
                      raise_exceptions=False)

=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [19]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

naive_df = naive_result.to_pandas()
adv_df   = adv_result.to_pandas()

def summary(df, label):
    cols = ["faithfulness", "answer_relevancy",
            "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))

                   Naive RAG  Advanced RAG
faithfulness           0.550         0.775
answer_relevancy       0.297         0.255
context_precision      0.642         0.792
context_recall         0.700         0.800

Delta (Advanced - Naive):
faithfulness         0.225
answer_relevancy    -0.042
context_precision    0.150
context_recall       0.100
dtype: float64


In [20]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

README.md: 0.00B [00:00, ?B/s]

mrc/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

mrc/validation-00000-of-00001.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17554 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5841 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

In [21]:
# TODO: 안내에 따라 context_docs 리스트를 만들어 보세요.
# 마지막에 len(context_docs) 와 context_docs[0].page_content[:200] 을 출력해서 확인.

import random
from langchain_core.documents import Document
context_docs = ds_klue.filter(lambda x: not x["is_impossible"]).shuffle(seed=42).select(range(300))


Filter:   0%|          | 0/5841 [00:00<?, ? examples/s]

In [22]:
# TODO: db_klue 를 만들되, 300k 토큰 한도를 피하기 위해 100개씩 batch 로 add_documents 하세요.
#   힌트:
#     db_klue = Chroma(embedding_function=embedding)
#     BATCH = 100
#     for i in range(0, len(context_docs), BATCH):
#         db_klue.add_documents(context_docs[i:i+BATCH])


In [23]:
# TODO: questions_klue, ground_truths_klue 를 만드세요. 각각 길이 20.



In [24]:
# TODO: RAG_PROMPT + naive_retriever_klue + naive_chain_klue 를 만들고
#       questions_klue[0] 으로 한 번 invoke 한 결과를 출력해 보세요.



In [25]:
# TODO: multi_query_retriever_klue 정의 + logging.INFO 설정
#       질문 1개로 .invoke() 한 결과 문서 개수를 출력.



In [26]:
# TODO: HYDE_PROMPT + hyde_retrieve_klue(question, k=3) 함수 정의
#       질문 1개로 호출해서 가상 답변과 검색된 첫 문서를 출력.



In [27]:
# TODO: reranker_klue = CrossEncoder("BAAI/bge-reranker-v2-m3")  (또는 다른 다국어 모델)
#       rerank_klue(query, docs, top_k=3) 함수를 정의해 보세요. (Step 4 rerank 와 동일 구조)



In [28]:
# TODO: advanced_rag_klue(question) 함수 정의 + 질문 1개로 답변/컨텍스트 확인



In [29]:
# TODO: Step 6/7 코드를 KLUE-MRC 변수(_klue)에 맞게 수정해서 평균 비교표를 출력하세요.



In [30]:
# TODO (선택): 질문 수를 늘려 같은 평가를 반복한 뒤 paired t-test 로 차이 검정



In [31]:
# ==========================================
# KLUE-MRC 도메인 이식 퀘스트 풀차징 원샷 코드 🚀
# ==========================================
import random
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from datasets import Dataset

print("⏳ [1/5] KLUE 데이터셋 정제 및 추출 중...")
# Step B: 정답 있는 데이터만 추리고 중복 제거
filtered_ds = ds_klue.filter(lambda x: not x["is_impossible"]).shuffle(seed=42).select(range(300))
unique_klue = {}
for ex in filtered_ds:
    if ex["context"] not in unique_klue:
        unique_klue[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique_klue.items()]

print(f"⏳ [2/5] VectorDB(Chroma)에 뉴스 기사 {len(context_docs)}개 적재 중... (시간 조금 걸림)")
# Step C: DB 적재 (배치 처리)
db_klue = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(context_docs), BATCH):
    db_klue.add_documents(context_docs[i:i+BATCH])

# Step D: 평가용 20개 추출
eval_samples_klue = list(filtered_ds)[:20]
questions_klue = [ex["question"] for ex in eval_samples_klue]
ground_truths_klue = [ex["answers"]["text"][0] for ex in eval_samples_klue]

print("⏳ [3/5] 검색기(Retriever) 및 RAG 체인 조립 중...")
# Step E ~ Step I: 각종 검색기 및 체인 세팅
naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})
naive_chain_klue = (
    {"context": naive_retriever_klue | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT | llm | StrOutputParser()
)

def rerank_klue(query, docs, top_k=3):
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

def advanced_rag_klue(question):
    candidates = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)
    top = rerank_klue(question, candidates, top_k=3)
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({"context": context, "question": question})
    return answer, top

print("⏳ [4/5] 파이프라인 답변 생성 중 (Naive & Advanced)...")
# 답변 및 컨텍스트 수집
naive_answers_k, naive_contexts_k = [], []
adv_answers_k, adv_contexts_k = [], []

for q in questions_klue:
    # Naive
    ctx = naive_retriever_klue.invoke(q)
    naive_answers_k.append(naive_chain_klue.invoke(q))
    naive_contexts_k.append([d.page_content for d in ctx])
    # Advanced
    a, ctx_adv = advanced_rag_klue(q)
    adv_answers_k.append(a)
    adv_contexts_k.append([d.page_content for d in ctx_adv])

# 데이터셋 만들기
naive_ds_klue = Dataset.from_dict({"user_input": questions_klue, "response": naive_answers_k, "retrieved_contexts": naive_contexts_k, "reference": ground_truths_klue})
adv_ds_klue = Dataset.from_dict({"user_input": questions_klue, "response": adv_answers_k, "retrieved_contexts": adv_contexts_k, "reference": ground_truths_klue})

print("⏳ [5/5] RAGAS 최종 채점 시작 (잠시만 대기해주세요!)...")
# Step J: RAGAS 채점 및 결과 출력
naive_res_klue = evaluate(naive_ds_klue, metrics=metrics, llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)
adv_res_klue = evaluate(adv_ds_klue, metrics=metrics, llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)

compare_klue = pd.concat([summary(naive_res_klue.to_pandas(), "Naive RAG (뉴스)"),
                          summary(adv_res_klue.to_pandas(), "Advanced RAG (뉴스)")], axis=1)

print("\n🎉 [완료] KLUE-MRC 도메인 이식 결과 비교표 🎉\n")
print(compare_klue.round(3))

⏳ [1/5] KLUE 데이터셋 정제 및 추출 중...


Filter:   0%|          | 0/5841 [00:00<?, ? examples/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


⏳ [2/5] VectorDB(Chroma)에 뉴스 기사 299개 적재 중... (시간 조금 걸림)
⏳ [3/5] 검색기(Retriever) 및 RAG 체인 조립 중...
⏳ [4/5] 파이프라인 답변 생성 중 (Naive & Advanced)...
⏳ [5/5] RAGAS 최종 채점 시작 (잠시만 대기해주세요!)...


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]


🎉 [완료] KLUE-MRC 도메인 이식 결과 비교표 🎉

                   Naive RAG (뉴스)  Advanced RAG (뉴스)
faithfulness                0.575              0.650
answer_relevancy            0.277              0.249
context_precision           0.483              0.700
context_recall              0.600              0.700


RAGAS 지표 결과 및 회고 (Naive vs Advanced)

(※ 테스트 환경: KLUE-MRC 샘플링 20건, gpt-4o-mini Judge)
Context Precision (0.483 ➔ 0.700): Reranker의 도입 효과가 가장 극적으로 나타난 지표입니다. 초기 검색된 10개의 넓은 문서 풀에서, 실제 정답이 포함된 문서를 상위 3개로 정확히 끌어올리는 데 성공했습니다.

Faithfulness (0.575 ➔ 0.650) & Context Recall (0.600 ➔ 0.700): 검색된 문맥의 질이 향상됨에 따라 답변의 사실 부합도와 정답 재현율이 동반 상승하는 것을 확인했습니다.

Answer Relevancy에 대한 고찰: 추출형(Extractive) QA 특성상 답변이 단답형으로 나오기 때문에 RAGAS가 질문을 역추론할 때 패널티를 받아 전반적인 수치가 낮게(0.2점대) 형성되었습니다. 이는 모델의 한계라기보다 데이터셋 구조적 특성으로 판단했습니다.